# Project: Credit Risk Classification

Compare Logistic Regression, Random Forest, and XGBoost for predicting loan default risk.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, classification_report)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print('Libraries loaded')

In [ ]:
np.random.seed(42)
n = 5000
df = pd.DataFrame({
    'credit_score': np.random.normal(650, 80, n).clip(300, 850).astype(int),
    'income': np.random.lognormal(10.8, 0.6, n).astype(int),
    'loan_amount': np.random.lognormal(9.5, 1, n).astype(int),
    'dti_ratio': np.random.uniform(0, 50, n),
    'employment_years': np.random.exponential(8, n).clip(0, 40).astype(int),
    'num_derogatories': np.random.poisson(0.5, n),
    'home_ownership': np.random.choice(['Rent','Mortgage','Own','Other'], n),
})
log_odds = -4 + 0.005*(700-df['credit_score']) + 0.0001*(100000-df['income']) +\
           0.00005*df['loan_amount'] + 0.02*df['dti_ratio'] +\
           0.3*df['num_derogatories'] + (df['home_ownership']=='Rent').astype(int)*0.3
df['default'] = (1/(1+np.exp(-(log_odds + np.random.normal(0,0.5,n)))) > 0.3).astype(int)
print(f'Default rate: {df["default"].mean():.2%}')
print(f'Dataset: {df.shape}')

In [ ]:
df_enc = pd.get_dummies(df, columns=['home_ownership'], drop_first=True)
X, y = df_enc.drop('default', axis=1), df_enc['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## Compare Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}
results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred, y_proba = model.predict(X_test_s), model.predict_proba(X_test_s)[:,1]
    results.append({'Model': name, 'Accuracy': accuracy_score(y_test, y_pred),
                    'Precision': precision_score(y_test, y_pred),
                    'Recall': recall_score(y_test, y_pred),
                    'F1': f1_score(y_test, y_pred),
                    'ROC-AUC': roc_auc_score(y_test, y_proba)})
print(pd.DataFrame(results).round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, model in models.items():
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test_s)[:,1])
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, model.predict_proba(X_test_s)[:,1]):.3f})')
axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
axes[0].legend()
axes[0].set_title('ROC Curves')
axes[0].grid(True, alpha=0.3)
rf = models['Random Forest']
imp = pd.DataFrame({'feature':X.columns, 'importance':rf.feature_importances_})\n    .sort_values('importance', ascending=True).tail(10)
axes[1].barh(imp['feature'], imp['importance'], color='steelblue')
axes[1].set_title('Top 10 Features (RF)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

- Synthetic credit risk data with realistic default patterns
- Compared Logistic Regression, Random Forest, XGBoost
- Evaluated with accuracy, precision, recall, F1, ROC-AUC
- Visualized ROC curves and feature importance